# Day 3: Feature Selection

**Theme:** Choose useful inputs for multiple linear regression while avoiding target leakage.

Today we will:

- load the Day 2 target dataset
- set `abs_g_mag` as the regression target
- define baseline and experimental feature sets
- document why `parallax`, `distance_pc`, and `phot_g_mean_mag` should not be ordinary predictors
- inspect feature correlations with the target
- plot feature-target relationships
- standardize selected numeric features before regression
- save a small Day 3 feature table for later notebooks

Multiple linear regression predicts one target from several input features:

```text
prediction = w1*x1 + w2*x2 + ... + b
```

Feature selection decides which `x` values the model is allowed to use. This matters because a model can look excellent for the wrong reason if we accidentally give it columns that directly contain the answer.


## 1. Imports

We use `StandardScaler` because features such as `bp_rp`, `parallax_snr`, `ra`, and `dec` live on different numeric scales.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", None)


## 2. Load The Day 2 Target Dataset

Day 3 normally starts from `gaia_day2_absolute_magnitude.csv`. If that file is missing, the notebook falls back to the shared cleaned Gaia loader and recomputes `distance_pc` and `abs_g_mag` using the Day 2 formulas.


In [ ]:
project_root_candidates = [Path(".."), Path("."), Path("gaia-explorer")]
PROJECT_ROOT = next((path for path in project_root_candidates if (path / "src").exists()), None)

if PROJECT_ROOT is None:
    searched = "\n".join(str(path.resolve()) for path in project_root_candidates)
    raise FileNotFoundError(f"Could not find gaia-explorer/src. Searched:\n{searched}")

sys.path.insert(0, str(PROJECT_ROOT.resolve()))

from src.data_clean import add_absolute_magnitude_target
from src.data_load import load_clean_gaia_sample

input_candidates = [
    Path("../data/processed/gaia_day2_absolute_magnitude.csv"),
    Path("data/processed/gaia_day2_absolute_magnitude.csv"),
    Path("gaia-explorer/data/processed/gaia_day2_absolute_magnitude.csv"),
]

DATA_PATH = next((path for path in input_candidates if path.exists()), None)

if DATA_PATH is not None:
    target_df = pd.read_csv(DATA_PATH)
    print(f"Loaded Day 2 target data from: {DATA_PATH}")
else:
    clean_df, clean_path = load_clean_gaia_sample()
    target_df = add_absolute_magnitude_target(clean_df)

    output_path = next((path for path in input_candidates if path.parent.exists()), input_candidates[0])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    target_df.to_csv(output_path, index=False)

    DATA_PATH = output_path
    print(f"Recomputed Day 2 target data from {clean_path} and saved it to: {DATA_PATH}")

print("Target dataset shape:", target_df.shape)
target_df.head()


## 3. Confirm Required Columns

Day 3 needs the target plus the candidate features. We keep `parallax` and `phot_g_mean_mag` available only so we can discuss leakage; they should not be ordinary predictors for `abs_g_mag`.


In [ ]:
required_columns = [
    "abs_g_mag",
    "bp_rp",
    "parallax_snr",
    "ra",
    "dec",
    "parallax",
    "distance_pc",
    "phot_g_mean_mag",
]

missing_required_columns = [col for col in required_columns if col not in target_df.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

modeling_df = target_df.dropna(subset=required_columns).copy()

print("Rows before Day 3 required-column filter:", len(target_df))
print("Rows after Day 3 required-column filter:", len(modeling_df))
modeling_df[required_columns].head()


## 4. Define The Target And Feature Sets

The target is:

```python
y = abs_g_mag
```

We will compare two feature sets:

- **Baseline:** `bp_rp`
- **Experimental:** `bp_rp`, `parallax_snr`, `ra`, `dec`

`bp_rp` is the cleanest physically motivated feature here because stellar color is tied to temperature and position on the color-magnitude diagram.

`parallax_snr` needs extra caution: it is not directly in the target formula, but it contains `parallax`, and `abs_g_mag` is computed from `parallax`. That makes it an indirect leakage-risk feature. We can use it for diagnostics and controlled experiments, but the clean baseline model should rely on `bp_rp` first.

`ra` and `dec` are also experimental diagnostics. They describe sky position, not the physical luminosity mechanism of a star.

### Physical Features Versus Diagnostic Features

A **physical feature** helps explain the star itself. For this project, `bp_rp` is physical because it describes color, and color is connected to stellar temperature and brightness.

A **diagnostic feature** helps us inspect data quality, bias, or model behavior. It can be useful for analysis, but we should be cautious about treating it as a real cause of brightness.

In this notebook:

- `bp_rp` is a physical feature.
- `parallax_snr` is a diagnostic measurement-quality feature.
- `ra` and `dec` are diagnostic sky-position features.
- `parallax`, `distance_pc`, and `phot_g_mean_mag` are leakage-risk features and should be excluded from ordinary predictors.

Diagnostic features are still useful because they help answer questions such as:

- Are low-quality measurements causing unusual points?
- Do model errors get worse when `parallax_snr` is low?
- Does the sample behave differently in different sky regions?


In [ ]:
y = modeling_df["abs_g_mag"]

baseline_features = ["bp_rp"]
experimental_features = ["bp_rp", "parallax_snr", "ra", "dec"]
selected_features = experimental_features

excluded_leakage_features = ["parallax", "distance_pc", "phot_g_mean_mag"]
indirect_leakage_risk_features = ["parallax_snr"]

feature_plan = pd.DataFrame(
    [
        {
            "group": "target",
            "columns": "abs_g_mag",
            "reason": "The value we want the model to predict.",
        },
        {
            "group": "baseline features",
            "columns": ", ".join(baseline_features),
            "reason": "Simple physical baseline using color only.",
        },
        {
            "group": "experimental features",
            "columns": ", ".join(experimental_features),
            "reason": "Diagnostic experiment only; parallax_snr contains parallax, so it carries indirect leakage risk.",
        },
        {
            "group": "indirect leakage-risk features",
            "columns": ", ".join(indirect_leakage_risk_features),
            "reason": "Useful for quality diagnostics, but not a clean physical predictor because abs_g_mag also depends on parallax.",
        },
        {
            "group": "excluded leakage features",
            "columns": ", ".join(excluded_leakage_features),
            "reason": "These columns are used directly, or indirectly, to compute abs_g_mag.",
        },
    ]
)

feature_plan


## 5. Target Leakage Note

Target leakage happens when a model receives a predictor that directly contains information used to create the target. It can make model scores look better than they really are.

For this project, `abs_g_mag` is computed from `phot_g_mean_mag` and `parallax`:

```text
abs_g_mag = phot_g_mean_mag + 5 * log10(parallax) - 10
```

So ordinary regression features should exclude:

- `phot_g_mean_mag`, because it is directly in the target formula
- `parallax`, because it is directly in the target formula
- `distance_pc`, because it is computed from `parallax`

`parallax_snr` is more subtle. It is computed from `parallax / parallax_error`, so it contains `parallax` even though it is mainly a measurement-quality column. For a clean physical model, treat `parallax_snr` as a diagnostic feature rather than a normal predictor. It is still useful to test experimentally because it can show whether measurement quality is affecting the target.

We can inspect those columns for understanding, but giving them to the model would leak the answer.


## 6. Correlation Diagnostics

Correlation is not proof of causation, but it is a useful first check. Values near `1` or `-1` mean a stronger linear relationship with `abs_g_mag`; values near `0` mean a weaker linear relationship.


In [ ]:
correlation_columns = experimental_features + ["abs_g_mag"]

correlation_table = (
    modeling_df[correlation_columns]
    .corr(numeric_only=True)["abs_g_mag"]
    .drop("abs_g_mag")
    .rename("correlation_with_abs_g_mag")
    .to_frame()
)
correlation_table["absolute_correlation"] = correlation_table["correlation_with_abs_g_mag"].abs()
correlation_table = correlation_table.sort_values("absolute_correlation", ascending=False)

correlation_table


In [ ]:
leakage_reference_columns = excluded_leakage_features + ["abs_g_mag"]

leakage_reference_table = (
    modeling_df[leakage_reference_columns]
    .corr(numeric_only=True)["abs_g_mag"]
    .drop("abs_g_mag")
    .rename("correlation_with_abs_g_mag")
    .to_frame()
)
leakage_reference_table["use_as_predictor"] = "No - target leakage risk"
leakage_reference_table


## 7. Plot `bp_rp` Versus `abs_g_mag`

This is the most important Day 3 plot. If `bp_rp` follows a visible pattern with `abs_g_mag`, then color is a reasonable baseline feature for predicting intrinsic brightness.

How to read the numbers:

- `bp_rp` near `0` means a bluer, hotter star.
- `bp_rp` near `1` means a more yellow-white-ish star.
- `bp_rp` near `2` or `3` means a redder, cooler star.
- `abs_g_mag` near `0` is intrinsically bright.
- `abs_g_mag` near `10` or `15` is intrinsically faint.

Why this plot is useful:

- The points form a visible curved diagonal band instead of a random cloud.
- As `bp_rp` gets larger, `abs_g_mag` generally gets larger too.
- Since larger magnitude means fainter, this says redder/cooler stars are usually fainter.
- That pattern is real astronomy structure: the main sequence.

Decision rule: a feature is more useful when its plot shows a clear pattern with the target. Here, `bp_rp` shows a clear color-brightness pattern, so it is a strong clean baseline feature.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

scatter = ax.scatter(
    modeling_df["bp_rp"],
    modeling_df["abs_g_mag"],
    c=np.log10(modeling_df["parallax_snr"]),
    cmap="viridis",
    s=12,
    alpha=0.65,
    linewidths=0,
)

ax.invert_yaxis()
ax.set_title("BP - RP Color Versus Absolute G Magnitude")
ax.set_xlabel("BP - RP color")
ax.set_ylabel("Absolute G magnitude (abs_g_mag)")

colorbar = plt.colorbar(scatter, ax=ax)
colorbar.set_label("log10(parallax_snr)")

plt.tight_layout()
plt.show()


## 8. Plot `parallax_snr` Versus `abs_g_mag`

`parallax_snr` is a measurement-quality feature. It may help diagnose unreliable points, but it is not the same kind of physical stellar property as color. Because it includes `parallax`, and `parallax` is used to compute `abs_g_mag`, it should not be treated as a clean predictor in the main model.

How to read the numbers:

- `parallax_snr = 5` is the minimum reliability we kept after cleaning.
- `parallax_snr = 10` is a decent parallax measurement.
- `parallax_snr = 30` is a good parallax measurement.
- `parallax_snr = 100` or higher is very reliable.
- The x-axis uses a log scale, so moving from `10` to `100` is a 10x jump.
- `abs_g_mag` still means intrinsic brightness: smaller values are brighter, larger values are fainter.

Why this plot is weaker as a predictor:

- The points look more like a broad cloud than a clean diagonal band.
- For the same `parallax_snr`, stars can have many different `abs_g_mag` values.
- That means knowing measurement reliability does not tell us much about intrinsic brightness.
- High SNR usually means the parallax is better measured, not that the star is brighter.

Decision rule: cloud-like plots usually indicate a weaker predictor. `parallax_snr` is useful for data quality and diagnostics, while `bp_rp` is better for predicting stellar brightness.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    modeling_df["parallax_snr"],
    modeling_df["abs_g_mag"],
    color="steelblue",
    s=12,
    alpha=0.55,
    linewidths=0,
)

ax.set_xscale("log")
ax.invert_yaxis()
ax.set_title("Parallax Signal-To-Noise Versus Absolute G Magnitude")
ax.set_xlabel("parallax_snr (log scale)")
ax.set_ylabel("Absolute G magnitude (abs_g_mag)")

plt.tight_layout()
plt.show()


## 9. Standardize Selected Features

Standardization converts each feature to roughly zero mean and unit standard deviation:

```text
scaled_value = (value - feature_mean) / feature_standard_deviation
```

This prevents large-scale columns such as `ra` and `dec` from numerically dominating smaller-scale columns such as `bp_rp`. Day 4 will fit the scaler on the training split before model evaluation; here we are preparing and inspecting the feature matrices.


In [ ]:
baseline_scaler = StandardScaler()
experimental_scaler = StandardScaler()

X_baseline_scaled = baseline_scaler.fit_transform(modeling_df[baseline_features])
X_experimental_scaled = experimental_scaler.fit_transform(modeling_df[experimental_features])

X_scaled = X_experimental_scaled
selected_feature_names = experimental_features

feature_matrix_summary = pd.DataFrame(
    [
        {
            "matrix": "baseline",
            "feature_names": ", ".join(baseline_features),
            "scaled_shape": str(X_baseline_scaled.shape),
        },
        {
            "matrix": "experimental",
            "feature_names": ", ".join(experimental_features),
            "scaled_shape": str(X_experimental_scaled.shape),
        },
    ]
)

feature_matrix_summary


In [ ]:
scaled_feature_preview = pd.DataFrame(
    X_experimental_scaled,
    columns=[f"scaled_{feature}" for feature in experimental_features],
    index=modeling_df.index,
)

scaled_feature_preview.head()


In [ ]:
scaled_feature_check = scaled_feature_preview.agg(["mean", "std"]).T
scaled_feature_check


## 10. Save The Day 3 Feature Dataset

The saved table keeps the target and selected non-leakage features together. Later notebooks can still redo scaling after a train/test split, which is the correct evaluation workflow.


In [ ]:
feature_output_candidates = [
    Path("../data/processed/gaia_day3_feature_selection.csv"),
    Path("data/processed/gaia_day3_feature_selection.csv"),
    Path("gaia-explorer/data/processed/gaia_day3_feature_selection.csv"),
]

FEATURE_OUTPUT_PATH = next(
    (path for path in feature_output_candidates if path.parent.exists()),
    feature_output_candidates[0],
)
FEATURE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

feature_dataset = modeling_df[["source_id", "abs_g_mag"] + selected_feature_names].copy()
feature_dataset.to_csv(FEATURE_OUTPUT_PATH, index=False)

print(f"Saved Day 3 feature data to: {FEATURE_OUTPUT_PATH}")
print("Saved shape:", feature_dataset.shape)
print("Selected feature names:", selected_feature_names)
print("Scaled experimental feature matrix shape:", X_experimental_scaled.shape)


## Reflection Questions And Starter Answers

1. Which feature seems most physically meaningful?

   **Starter answer:** `bp_rp` is the most physically meaningful feature because it measures color, and color is connected to stellar temperature. Temperature and intrinsic brightness are linked through the color-magnitude diagram.

2. Why would `phot_g_mean_mag` or `parallax` create data leakage for this target?

   **Starter answer:** `abs_g_mag` is computed directly from `phot_g_mean_mag` and `parallax`, so using either one as a normal predictor gives the model part of the answer. The model would look better than it really is at learning from independent stellar features.

3. Are `ra` and `dec` physically meaningful predictors of luminosity, or mostly diagnostic features?

   **Starter answer:** `ra` and `dec` are sky-position coordinates, not direct physical causes of a star's luminosity. They can be useful diagnostics if the sample has sky-region selection effects, but we should be cautious about interpreting them as physical predictors.

4. Why do we scale the feature matrix before regression?

   **Starter answer:** Scaling puts features with different units and ranges onto comparable numeric scales. This is especially important for gradient descent and helps coefficients become easier to compare.
